# 02 — Data Cleaning (Phase 2)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 2: import และเรียกใช้ฟังก์ชันจาก `src/rri/cleaning/`
เท่านั้น — ไม่มี business logic ในเซลล์ตาม CLAUDE.md N2 ลำดับขั้นตอนคงที่ตามที่ตกลงไว้:

1. Extract + structural fixes (encoding, BOM, header typo, twin-column coalesce, BE→CE)
2. Harmonize category taxonomy (**ก่อน** dedup)
3. Resolve dates ← **CHECKPOINT**
4. Deduplicate
5. Remaining field-level fixes (coordinates, times, injury policy, nulls)
6. Validate + write staging

อ้างอิงข้อเท็จจริงและเหตุผลทั้งหมดจาก Phase 1: [`CLAUDE.md`](../CLAUDE.md) §6 และ
[`docs/data_quality_findings.md`](../docs/data_quality_findings.md)

## 0. Setup

In [1]:
import sys
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

import polars as pl

from config.settings import settings
from rri.cleaning.dates import (
    apply_date_certainty,
    check_year_match,
    day_of_month_histogram,
    parse_dates,
    resolve_transposed_pairs_by_likelihood,
)
from rri.cleaning.dedup import deduplicate_accidents
from rri.cleaning.extract import (
    add_extract_metadata,
    coalesce_twin_columns,
    convert_buddhist_year,
    pivot_rainfall_leads,
    read_raw_csv,
    rename_headers,
)
from rri.cleaning.fields import (
    add_injury_breakdown_flag,
    clean_coordinates,
    convert_times,
)
from rri.cleaning.log import CleaningLogEntry, cleaning_log_to_frame, write_cleaning_log
from rri.cleaning.taxonomy import apply_taxonomy, load_taxonomy_mapping
from rri.contracts.raw import (
    check_gate1_reconciliation,
    check_gate2_reconciliation,
    check_injury_totals,
    run_accidents_raw_contract,
)
from rri.utils.logging import configure_logging
from rri.utils.profiling import (
    SOURCE_FILE_DATE_FORMAT,
    check_duplicate_date_string_consistency,
    compute_report_lag_days,
    find_conflicting_groups,
    find_transposed_date_string_pairs,
    report_lag_quantiles_by_group,
)

configure_logging()
settings.ensure_writable_dirs()
RUN_TIMESTAMP = datetime.now(UTC)
cleaning_log: list[CleaningLogEntry] = []
settings.raw_dir

PosixPath('/Users/phidphew/Documents/KU Work/Pee 3/Data Mining/group_project/project/data/00_raw')

## 1. Extract + structural fixes

เริ่มจาก extract ไฟล์ CSV ทั้ง 6 ไฟล์ (ไฟล์ accidents หลักจะผ่านอีก 5 ขั้นตอนถัดไป
ส่วนอีก 5 ไฟล์จบที่ขั้นตอนนี้) ทุกไฟล์อ่านด้วย `read_raw_csv` (encoding ตาม P7.2 —
BOM 5 ใน 6 ไฟล์, `stt_car_fuel` ไม่มี) แล้วเติม `_source_file`/`_ingested_at`/`_row_hash`

In [2]:
accidents_raw = read_raw_csv(settings.raw_dir / "thailand_accidents_2019_2026_combined.csv")
accidents_raw = add_extract_metadata(
    accidents_raw, "thailand_accidents_2019_2026_combined.csv", RUN_TIMESTAMP
)

accidents_step1, twin_logs = coalesce_twin_columns(accidents_raw)

for log_entry in twin_logs:
    cleaning_log.append(
        CleaningLogEntry(
            step="1_extract",
            rule=f"coalesce_twin_columns:{log_entry.old_column}",
            rows_before=log_entry.total_rows,
            rows_affected=log_entry.rows_from_new,
            description=(
                f"{log_entry.old_column} <- {log_entry.new_column}: "
                f"{log_entry.rows_from_old} from old, {log_entry.rows_from_new} from new, "
                f"{log_entry.rows_still_null} still null"
            ),
        )
    )

cleaning_log_to_frame(cleaning_log)

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""1_extract""","""coalesce_twin_columns:รถยนต์นั…",161307,8738,"""รถยนต์นั่งส่วนบุคคล <- รถยนต์น…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกไม่เกิน10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกมากกว่า10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถอื่นๆ""",161307,8738,"""รถอื่นๆ <- อื่นๆ: 133375 from …"


**ยืนยันตรงกับ Phase 1**: overlap ระหว่างคอลัมน์เก่า/ใหม่ทั้ง 4 คู่ = 0 แถว (ถ้าไม่ตรง
`coalesce_twin_columns` จะ raise ทันที ไม่ใช่ coalesce มั่ว ๆ) จำนวนแถวที่มาจากคอลัมน์ใหม่ต่อคู่
ตรงกับสัดส่วน 5.4% ที่ Phase 1 พบ

In [3]:
rainfall_raw = read_raw_csv(settings.raw_dir / "thailand_rainfall_2021_2026_combined.csv")
rainfall_raw = add_extract_metadata(
    rainfall_raw, "thailand_rainfall_2021_2026_combined.csv", RUN_TIMESTAMP
)
rainfall_renamed = rename_headers(rainfall_raw, {"Foreast_Month": "Forecast_Month"})

rainfall_step1 = pivot_rainfall_leads(rainfall_renamed)
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "pivot_rainfall_leads",
        rainfall_renamed.height,
        rainfall_step1.height,
        f"{rainfall_renamed.height} rows (6 leads x group) -> "
        f"{rainfall_step1.height} groups x rain_lead0..5",
    )
)
pl.DataFrame(
    [
        {
            "rows_before_pivot": rainfall_renamed.height,
            "rows_after_pivot": rainfall_step1.height,
            "columns": rainfall_step1.columns,
        }
    ]
)

rows_before_pivot,rows_after_pivot,columns
i64,i64,list[str]
30954,5159,"[""Initial_Year"", ""Initial_Month"", … ""rain_lead5""]"


**ไม่มีแถวหายไป**: `rows_before_pivot / 6 == rows_after_pivot` (6 lead-time ต่อกลุ่มกลายเป็น
1 แถว x 6 คอลัมน์ ไม่ใช่การ dedup) ยืนยันตาม P7.5

In [4]:
registration_raw = read_raw_csv(settings.raw_dir / "car_registered_new_2565_2569_combined.csv")
registration_raw = add_extract_metadata(
    registration_raw, "car_registered_new_2565_2569_combined.csv", RUN_TIMESTAMP
)
registration_step1 = convert_buddhist_year(registration_raw, "ปี พ.ศ.")
cleaning_log.append(
    CleaningLogEntry(
        "1_extract",
        "convert_buddhist_year",
        registration_step1.height,
        registration_step1.height,
        "added year_ce = ปี พ.ศ. - 543 for every row (original column kept)",
    )
)
registration_step1.select("ปี พ.ศ.", "year_ce").unique().sort("ปี พ.ศ.")

ปี พ.ศ.,year_ce
str,i64
"""2565""",2022
"""2566""",2023
"""2567""",2024
"""2568""",2025
"""2569""",2026


**ตรวจแล้ว**: พ.ศ. 2565–2569 กลายเป็น ค.ศ. 2022–2026 ตรงตามสูตร `CE = BE - 543`
คอลัมน์เดิมยังอยู่ครบ (ไม่ทำลายข้อมูลดิบ)

In [5]:
flood_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_flood_risk_areas.csv"),
    "thailand_flood_risk_areas.csv",
    RUN_TIMESTAMP,
)
heavy_rain_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "thailand_heavy_rain_R35mm.csv"),
    "thailand_heavy_rain_R35mm.csv",
    RUN_TIMESTAMP,
)
stt_car_fuel_raw = add_extract_metadata(
    read_raw_csv(settings.raw_dir / "stt_car_fuel_at_25690228.csv"),
    "stt_car_fuel_at_25690228.csv",
    RUN_TIMESTAMP,
)

pl.DataFrame(
    [
        {"file": "flood_risk", "rows": flood_raw.height, "cols": flood_raw.width},
        {"file": "heavy_rain_r35mm", "rows": heavy_rain_raw.height, "cols": heavy_rain_raw.width},
        {"file": "stt_car_fuel", "rows": stt_car_fuel_raw.height, "cols": stt_car_fuel_raw.width},
    ]
)

file,rows,cols
str,i64,i64
"""flood_risk""",27023,16
"""heavy_rain_r35mm""",3465,7
"""stt_car_fuel""",4643,8


ไฟล์อีก 3 ไฟล์นี้ไม่มี structural fix เฉพาะใน Step 1 (นอกจาก encoding/metadata) —
เขียนลง staging ตรง ๆ

In [6]:
accidents_step1.write_parquet(settings.staging_dir / "accidents.parquet")
rainfall_step1.write_parquet(settings.staging_dir / "rainfall.parquet")
registration_step1.write_parquet(settings.staging_dir / "car_registered_new.parquet")
flood_raw.write_parquet(settings.staging_dir / "flood_risk.parquet")
heavy_rain_raw.write_parquet(settings.staging_dir / "heavy_rain_r35mm.parquet")
stt_car_fuel_raw.write_parquet(settings.staging_dir / "stt_car_fuel.parquet")

sorted(p.name for p in settings.staging_dir.glob("*.parquet"))

['accidents.parquet',
 'car_registered_new.parquet',
 'flood_risk.parquet',
 'heavy_rain_r35mm.parquet',
 'rainfall.parquet',
 'stt_car_fuel.parquet']

**Step 1 เสร็จสมบูรณ์** — เขียนไฟล์ทั้ง 6 ลง `data/10_staging/` แล้ว `accidents.parquet`
จะถูกเขียนทับอีกครั้งใน Step 6 หลังผ่าน Step 2–5 ครบ ไฟล์ที่เหลือจบที่นี่ (ไม่มีขั้นตอนถัดไปแตะต้อง)

## 2. Harmonize category taxonomy (ก่อน dedup)

โหลด `accidents.parquet` จาก Step 1 แล้ว apply mapping table ทั้ง 3 ตัว
(`config/mappings/{cause,crash_type,road_geometry}_groups.csv` — สร้างโดย
`scripts/build_taxonomy_mappings.py`, ตรวจสอบด้วยมือแล้ว) **ต้องทำก่อน Step 4 (dedup)**
เพราะ P5 พบว่า 74% ของ "conflict" ในคู่ duplicate ที่คอลัมน์ `บริเวณที่เกิดเหตุ` จริง ๆ
เป็นแค่คำพ้องความหมายคนละชุด ไม่ใช่ conflict จริง

In [7]:
accidents_step1 = pl.read_parquet(settings.staging_dir / "accidents.parquet")

cause_mapping = load_taxonomy_mapping(settings.mappings_dir / "cause_groups.csv")
crash_type_mapping = load_taxonomy_mapping(settings.mappings_dir / "crash_type_groups.csv")
road_geometry_mapping = load_taxonomy_mapping(settings.mappings_dir / "road_geometry_groups.csv")

accidents_step2, cause_log = apply_taxonomy(accidents_step1, "มูลเหตุสันนิษฐาน", cause_mapping)
accidents_step2, crash_type_log = apply_taxonomy(
    accidents_step2, "ลักษณะการเกิดเหตุ", crash_type_mapping
)
accidents_step2, geometry_log = apply_taxonomy(
    accidents_step2, "บริเวณที่เกิดเหตุ", road_geometry_mapping
)
accidents_step2, geometry_secondary_log = apply_taxonomy(
    accidents_step2,
    "บริเวณที่เกิดเหตุอื่น",
    road_geometry_mapping,
    new_column="บริเวณที่เกิดเหตุอื่น_group",
)

for taxonomy_log in (cause_log, crash_type_log, geometry_log, geometry_secondary_log):
    cleaning_log.append(
        CleaningLogEntry(
            "2_taxonomy",
            f"apply_taxonomy:{taxonomy_log.column}",
            taxonomy_log.total_rows,
            taxonomy_log.rows_mapped,
            f"mapped={taxonomy_log.rows_mapped} null={taxonomy_log.rows_null} "
            f"unmapped={taxonomy_log.rows_unmapped} {taxonomy_log.unmapped_values[:5]}",
        )
    )

pl.DataFrame(
    [
        {
            "column": log_entry.column,
            "rows_mapped": log_entry.rows_mapped,
            "rows_null": log_entry.rows_null,
            "rows_unmapped": log_entry.rows_unmapped,
            "unmapped_sample": log_entry.unmapped_values[:5],
        }
        for log_entry in (cause_log, crash_type_log, geometry_log, geometry_secondary_log)
    ]
)

column,rows_mapped,rows_null,rows_unmapped,unmapped_sample
str,i64,i64,i64,list[null]
"""มูลเหตุสันนิษฐาน""",158627,2680,0,[]
"""ลักษณะการเกิดเหตุ""",153244,8063,0,[]
"""บริเวณที่เกิดเหตุ""",147866,13441,0,[]
"""บริเวณที่เกิดเหตุอื่น""",8462,152845,0,[]


**ตรวจ**: `rows_unmapped` ต้องเป็น 0 ทุกคอลัมน์ (mapping table ครอบคลุมทุกค่าที่พบจริงใน
ข้อมูล ณ ตอนสร้างตาราง) ถ้าไม่เป็น 0 แปลว่าข้อมูลจริงมีค่าที่ mapping table ไม่รู้จัก — ต้อง
หยุดแล้วรายงาน ไม่ใช่เดาแล้วปล่อยผ่าน

In [8]:
geometry_conflicts_before = find_conflicting_groups(accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ")
geometry_conflicts_after = find_conflicting_groups(
    accidents_step2, ["ACC_CODE"], "บริเวณที่เกิดเหตุ_group"
)

dup_groups = (
    accidents_step2.filter(pl.col("ACC_CODE").is_not_null())
    .group_by("ACC_CODE")
    .len()
    .filter(pl.col("len") > 1)
)

pl.DataFrame(
    [
        {
            "duplicate_groups_checked": dup_groups.height,
            "conflicts_before_harmonization": len(geometry_conflicts_before),
            "conflict_rate_before_pct": round(
                len(geometry_conflicts_before) / dup_groups.height * 100, 1
            ),
            "conflicts_after_harmonization": len(geometry_conflicts_after),
            "conflict_rate_after_pct": round(
                len(geometry_conflicts_after) / dup_groups.height * 100, 1
            ),
        }
    ]
)

duplicate_groups_checked,conflicts_before_harmonization,conflict_rate_before_pct,conflicts_after_harmonization,conflict_rate_after_pct
i64,i64,f64,i64,f64
1641,1216,74.1,52,3.2


**เทียบกับ Phase 1**: conflict rate เดิม (raw `บริเวณที่เกิดเหตุ`) ควรตรงกับ P5 ที่ 74%
(1,216/1,641) หลัง harmonize ด้วย `บริเวณที่เกิดเหตุ_group` (group ตามประเภทถนน ตัด slope
descriptor ทิ้ง) ควรลดลงเหลือแค่ residual ที่เป็น conflict จริง (ประเภทถนนต่างกันจริง ไม่ใช่แค่
คำอธิบายความชันต่างกัน)

In [9]:
accidents_step2.write_parquet(settings.staging_dir / "accidents.parquet")
cleaning_log_to_frame(cleaning_log).filter(pl.col("step") == "2_taxonomy")

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""2_taxonomy""","""apply_taxonomy:มูลเหตุสันนิษฐา…",161307,158627,"""mapped=158627 null=2680 unmapp…"
"""2_taxonomy""","""apply_taxonomy:ลักษณะการเกิดเห…",161307,153244,"""mapped=153244 null=8063 unmapp…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,147866,"""mapped=147866 null=13441 unmap…"
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,8462,"""mapped=8462 null=152845 unmapp…"


**Step 2 เสร็จสมบูรณ์** — เขียนทับ `accidents.parquet` ด้วยเวอร์ชันที่มี taxonomy
harmonized แล้ว (`_group` columns เพิ่มเข้ามา คอลัมน์ raw เดิมยังอยู่ครบ) พร้อมสำหรับ Step 3

## 3. Resolve dates — CHECKPOINT

โหลด `accidents.parquet` จาก Step 2 แล้ว parse วันที่ตามรูปแบบต่อไฟล์ (P1) จากนั้นตรวจ
safety net ทั้ง 4 ข้อ และแก้ 448 คู่ transposed-date ด้วย **likelihood discriminator ต่อหน่วยงาน**
(แทนที่ binary threshold test เดิมของ Phase 1) — เทียบ lag ของแต่ละตัวเลือกกับค่ามัธยฐานของ
หน่วยงานนั้น เลือกตัวที่ใกล้กว่า แล้วบันทึก margin ไว้เป็นตัวชี้วัดความมั่นใจ

In [10]:
accidents_step2 = pl.read_parquet(settings.staging_dir / "accidents.parquet")
accidents_step3 = parse_dates(accidents_step2, SOURCE_FILE_DATE_FORMAT)

pl.DataFrame(
    [
        {
            "rows": accidents_step3.height,
            "accident_date_null": accidents_step3["accident_date"].null_count(),
            "report_date_null": accidents_step3["report_date"].null_count(),
        }
    ]
)

rows,accident_date_null,report_date_null
i64,i64,i64
161307,1,82


In [11]:
year_match = check_year_match(accidents_step3)

pl.DataFrame(
    [
        {
            "rows_checked": year_match.rows_checked,
            "rows_unparseable": year_match.rows_unparseable,
            "rows_mismatched": year_match.rows_mismatched,
        }
    ]
)

rows_checked,rows_unparseable,rows_mismatched
i64,i64,i64
161307,1,0


**Safety net #1**: ปีที่ parse ได้ต้องตรงกับคอลัมน์ `ปีที่เกิดเหตุ` เดิมทุกแถว — mismatch ต้องเป็น 0
(ถ้าไม่ใช่ 0 ต้องหยุดและรายงานทันที ไม่ใช่ปรับตัวเลขให้ผ่าน)

In [12]:
safety_net_4a = check_duplicate_date_string_consistency(
    accidents_step3, "ACC_CODE", "วันที่เกิดเหตุ", "accident_date"
)

pl.DataFrame(
    [
        {
            "groups_checked (identical raw string)": safety_net_4a.groups_checked,
            "mismatched_groups": safety_net_4a.mismatched_groups,
        }
    ]
)

groups_checked (identical raw string),mismatched_groups
i64,i64
1193,0


**Safety net #4a**: กลุ่ม duplicate ที่ raw string เหมือนกันเป๊ะต้อง parse ได้วันที่เดียวกันเสมอ —
mismatch ต้องเป็น 0 (ตรงกับที่ Phase 1 ยืนยันไว้แล้วที่ 1,193 กลุ่ม)

In [13]:
transposed_keys = find_transposed_date_string_pairs(
    accidents_step3.filter(pl.col("ACC_CODE").is_not_null()), "ACC_CODE", "วันที่เกิดเหตุ"
)

lagged = compute_report_lag_days(
    accidents_step3, "source_file", "วันที่เกิดเหตุ", "วันที่รายงาน", SOURCE_FILE_DATE_FORMAT
)
agency_lag_stats = report_lag_quantiles_by_group(lagged, "หน่วยงาน")
agency_median_lag = {row["หน่วยงาน"]: row["median_days"] for row in agency_lag_stats.to_dicts()}
agency_max_lag = {row["หน่วยงาน"]: float(row["max_days"]) for row in agency_lag_stats.to_dicts()}

likelihood_resolutions = resolve_transposed_pairs_by_likelihood(
    accidents_step3.filter(pl.col("ACC_CODE").is_not_null()),
    "ACC_CODE",
    transposed_keys,
    "วันที่เกิดเหตุ",
    "วันที่รายงาน",
    "หน่วยงาน",
    month_first=True,
    agency_median_lag_days=agency_median_lag,
    agency_max_lag_days=agency_max_lag,
)

resolved_count = sum(1 for r in likelihood_resolutions if r.outcome == "resolved")
ambiguous_count = sum(1 for r in likelihood_resolutions if r.outcome == "ambiguous")
neither_viable_count = sum(
    1 for r in likelihood_resolutions if r.outcome == "ambiguous" and r.winner_date is None
)
order_violation_count = sum(1 for r in likelihood_resolutions if r.had_order_violation)

pl.DataFrame(
    [
        {
            "transposed_pairs_found": len(transposed_keys),
            "resolved": resolved_count,
            "ambiguous": ambiguous_count,
            "sum": resolved_count + ambiguous_count,
            "ambiguous_both_hard_valid": ambiguous_count - neither_viable_count,
            "ambiguous_neither_hard_valid": neither_viable_count,
            "groups_with_an_order_violation_elimination": order_violation_count,
        }
    ]
)

transposed_pairs_found,resolved,ambiguous,sum,ambiguous_both_hard_valid,ambiguous_neither_hard_valid,groups_with_an_order_violation_elimination
i64,i64,i64,i64,i64,i64,i64
448,321,127,448,112,15,336


**แก้ไขแล้ว (owner-requested review): เดิม likelihood อย่างเดียวทำให้ 448/448 resolved 0 ambiguous — เป็นสัญญาณเตือน ไม่ใช่ชัยชนะ** เพราะไม่มี hard filter คัดคู่ที่ผิดกฎ `accident_date ≤ report_date` ออกก่อน ตอนนี้ใช้ 2 ขั้น: **ขั้น 1 (hard filter)** คัดคู่ที่ผิดกฎ chronology ออกก่อน (เทียบกับ **ทุก** วันที่รายงานที่พบในกลุ่ม ไม่ใช่แค่คู่ของแถวตัวเอง — เพราะวันที่รายงานเองก็สลับตำแหน่งได้เหมือนกัน ตาม P5) **ขั้น 2 (likelihood)** ใช้ตัดสินเฉพาะเมื่อเหลือ 2 ตัวเลือกหลัง hard filter เท่านั้น — และคู่แบบนั้นยังคง `date_certainty = ambiguous` เสมอ (แม้จะบันทึก winner ไว้อ้างอิงก็ตาม) เพราะเป็นการเดา ไม่ใช่ข้อสรุปที่บังคับได้ทางตรรกะ

In [14]:
accidents_step3 = apply_date_certainty(accidents_step3, likelihood_resolutions, "ACC_CODE")

accidents_step3["date_certainty"].value_counts()

date_certainty,count
str,u32
"""certain""",160711
"""ambiguous""",596


In [15]:
histogram = day_of_month_histogram(accidents_step3)
pl.DataFrame({"day": list(histogram.keys()), "count": list(histogram.values())})

day,count
i64,i64
1,7124
2,5682
3,5400
4,5610
5,4960
…,…
27,5177
28,5467
29,5630


**Safety net #3**: กระจายตัวของวันที่ (day-of-month) ควรค่อนข้างสม่ำเสมอตลอด 1-31 (มีลดลงบ้างที่
29-31 ตามธรรมชาติของจำนวนวันในเดือน) ถ้า parse ผิดจะเห็น spike ที่วัน 1-12 ชัดเจน (เพราะ MDY/DMY สลับกัน
จะโดนบีบเข้าช่วง 1-12 เท่านั้น)

In [16]:
sample_keys = [r.group_key for r in likelihood_resolutions if r.outcome == "resolved"][:5]

rows = []
for key in sample_keys:
    resolution = next(r for r in likelihood_resolutions if r.group_key == key)
    sub = accidents_step3.filter(pl.col("ACC_CODE") == key).select(
        "ACC_CODE", "วันที่เกิดเหตุ", "วันที่รายงาน", "หน่วยงาน"
    )
    for row in sub.to_dicts():
        rows.append(
            {
                "ACC_CODE": row["ACC_CODE"],
                "raw_date": row["วันที่เกิดเหตุ"],
                "report_date_raw": row["วันที่รายงาน"],
                "agency": row["หน่วยงาน"],
                "winner_date": resolution.winner_date,
                "outcome": resolution.outcome,
                "had_order_violation": resolution.had_order_violation,
            }
        )
pl.DataFrame(rows)

ACC_CODE,raw_date,report_date_raw,agency,winner_date,outcome,had_order_violation
str,str,str,str,date,str,bool
"""7642011""","""5/7/2023""","""5/8/2023""","""กรมทางหลวงชนบท""",2023-05-07,"""resolved""",true
"""7642011""","""7/5/2023""","""8/5/2023""","""กรมทางหลวงชนบท""",2023-05-07,"""resolved""",true
"""7606906""","""2/6/2023""","""3/16/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-02-06,"""resolved""",true
"""7606906""","""6/2/2023""","""3/16/2023""","""การทางพิเศษแห่งประเทศไทย""",2023-02-06,"""resolved""",true
"""7645202""","""5/10/2023""","""5/12/2023""","""กรมทางหลวงชนบท""",2023-05-10,"""resolved""",true
"""7645202""","""10/5/2023""","""12/5/2023""","""กรมทางหลวงชนบท""",2023-05-10,"""resolved""",true
"""7544351""","""1/3/2023""","""1/4/2023""","""กรมทางหลวงชนบท""",2023-01-03,"""resolved""",true
"""7544351""","""3/1/2023""","""4/1/2023""","""กรมทางหลวงชนบท""",2023-01-03,"""resolved""",true
"""7544356""","""1/3/2023""","""1/4/2023""","""กรมทางหลวงชนบท""",2023-01-03,"""resolved""",true


**ตัวอย่างคู่ที่ยังเหลือ `ambiguous` (ทั้งสองตัวเลือกผ่าน hard filter จริง ไม่ใช่ discriminator เดา):**

In [17]:
ambiguous_sample_keys = [
    r.group_key
    for r in likelihood_resolutions
    if r.outcome == "ambiguous" and r.winner_date is not None
][:5]

rows = []
for key in ambiguous_sample_keys:
    resolution = next(r for r in likelihood_resolutions if r.group_key == key)
    sub = accidents_step3.filter(pl.col("ACC_CODE") == key).select(
        "ACC_CODE", "วันที่เกิดเหตุ", "วันที่รายงาน", "หน่วยงาน"
    )
    for row in sub.to_dicts():
        rows.append(
            {
                "ACC_CODE": row["ACC_CODE"],
                "raw_date": row["วันที่เกิดเหตุ"],
                "report_date_raw": row["วันที่รายงาน"],
                "agency": row["หน่วยงาน"],
                "likelihood_winner": resolution.winner_date,
                "margin_days": resolution.margin_days,
                "outcome": resolution.outcome,
            }
        )
pl.DataFrame(rows)

ACC_CODE,raw_date,report_date_raw,agency,likelihood_winner,margin_days,outcome
str,str,str,str,date,f64,str
"""8063722""","""8/9/2023""","""11/9/2023""","""กรมทางหลวงชนบท""",2023-09-08,89.0,"""ambiguous"""
"""8063722""","""9/8/2023""","""9/11/2023""","""กรมทางหลวงชนบท""",2023-09-08,89.0,"""ambiguous"""
"""8740193""","""3/12/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-12-03,266.0,"""ambiguous"""
"""8740193""","""12/3/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-12-03,266.0,"""ambiguous"""
"""9233296""","""4/11/2023""","""1/2/2024""","""กรมทางหลวงชนบท""",2023-11-04,177.0,"""ambiguous"""
…,…,…,…,…,…,…
"""8740363""","""6/7/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-07-06,29.0,"""ambiguous"""
"""8740363""","""7/6/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-07-06,29.0,"""ambiguous"""
"""8740363""","""7/6/2023""","""12/19/2023""","""กรมทางหลวงชนบท""",2023-07-06,29.0,"""ambiguous"""


ทั้งสองตัวเลือกผ่าน hard filter (accident ≤ report ทุกวันที่รายงานในกลุ่ม, gap ≤ ceiling) จริง —
`likelihood_winner` บันทึกไว้อ้างอิงเท่านั้น `date_certainty` ของทั้งสองแถวเป็น `ambiguous` เสมอ

In [18]:
accidents_step3.write_parquet(settings.staging_dir / "accidents.parquet")
"step3 written"

'step3 written'

## 4. Deduplicate

โหลด `accidents.parquet` จาก Step 3 (มี `date_certainty` แล้ว) แล้ว dedup ตาม `ACC_CODE`
ด้วยกฎที่ตกลงไว้: certain > ambiguous → null น้อยกว่า → ACC_CODE ไม่ null → `_row_hash` (deterministic)

In [19]:
accidents_step3 = pl.read_parquet(settings.staging_dir / "accidents.parquet")

DERIVED_COLUMNS = {
    "_source_file",
    "_ingested_at",
    "_row_hash",
    "มูลเหตุสันนิษฐาน_group",
    "ลักษณะการเกิดเหตุ_group",
    "บริเวณที่เกิดเหตุ_group",
    "บริเวณที่เกิดเหตุอื่น_group",
    "accident_date",
    "report_date",
    "date_certainty",
}
original_columns = [c for c in accidents_step3.columns if c not in DERIVED_COLUMNS]

accidents_step4, dedup_result = deduplicate_accidents(
    accidents_step3, columns_for_null_count=original_columns
)

cleaning_log.append(
    CleaningLogEntry(
        "4_dedup",
        "deduplicate_accidents",
        dedup_result.rows_before,
        dedup_result.rows_before - dedup_result.rows_after,
        f"{dedup_result.groups_collapsed} ACC_CODE groups collapsed to 1 row each "
        f"({dedup_result.rows_before - dedup_result.rows_after} excess rows removed), "
        f"{dedup_result.rows_before} -> {dedup_result.rows_after} rows",
    )
)

pl.DataFrame(
    [
        {
            "rows_before": dedup_result.rows_before,
            "rows_after": dedup_result.rows_after,
            "groups_collapsed": dedup_result.groups_collapsed,
            "fatalities_before": accidents_step3["ผู้เสียชีวิต"]
            .cast(pl.Float64, strict=False)
            .fill_null(0)
            .sum(),
            "fatalities_after": accidents_step4["ผู้เสียชีวิต"]
            .cast(pl.Float64, strict=False)
            .fill_null(0)
            .sum(),
        }
    ]
)

2026-08-14T03:06:54.887870Z [info     ] accidents_deduplicated         examples=[{'group_key': '7537563', 'rows_in_group': 2, 'kept_certainty': 'certain', 'dropped_certainties': ('certain',)}, {'group_key': '7739744', 'rows_in_group': 2, 'kept_certainty': 'certain', 'dropped_certainties': ('certain',)}, {'group_key': '8902329', 'rows_in_group': 2, 'kept_certainty': 'certain', 'dropped_certainties': ('certain',)}, {'group_key': '7634468', 'rows_in_group': 2, 'kept_certainty': 'certain', 'dropped_certainties': ('certain',)}, {'group_key': '8961103', 'rows_in_group': 2, 'kept_certainty': 'ambiguous', 'dropped_certainties': ('ambiguous',)}] groups_collapsed=1641 rows_after=159605 rows_before=161307 run_id=2a5e957b166e45ff9677162dbb0db424


rows_before,rows_after,groups_collapsed,fatalities_before,fatalities_after
i64,i64,i64,f64,f64
161307,159605,1641,21814.0,21497.0


**ยืนยันตรงกับ CLAUDE.md P5**: 1,641 กลุ่มยุบเหลือกลุ่มละ 1 แถว (ตัดแถวส่วนเกิน 1,702 แถวออก), ยอดผู้เสียชีวิตควรเปลี่ยนจาก 21,814 →
21,497 (Gate 1 → Gate 2) `assert_casualty_agreement` ผ่านแล้วโดยอัตโนมัติก่อน dedup (raise ทันทีถ้าไม่ตรง)

In [20]:
pl.DataFrame(
    [
        {
            "ACC_CODE": e.group_key,
            "rows_in_group": e.rows_in_group,
            "kept_certainty": e.kept_certainty,
            "dropped_certainties": e.dropped_certainties,
        }
        for e in dedup_result.examples
    ]
)

ACC_CODE,rows_in_group,kept_certainty,dropped_certainties
str,i64,str,list[str]
"""7537563""",2,"""certain""","[""certain""]"
"""7739744""",2,"""certain""","[""certain""]"
"""8902329""",2,"""certain""","[""certain""]"
"""7634468""",2,"""certain""","[""certain""]"
"""8961103""",2,"""ambiguous""","[""ambiguous""]"


In [21]:
accidents_step4.write_parquet(settings.staging_dir / "accidents.parquet")
"step4 written"

'step4 written'

## 5. Remaining field-level fixes

โหลด `accidents.parquet` จาก Step 4 แล้วแก้ 3 เรื่องที่เหลือ (แยกอิสระจากกัน ไม่ต้อง impute
ใดๆ): พิกัด, รูปแบบเวลา, และ flag ความพร้อมของข้อมูลบาดเจ็บแยกสาหัส/เล็กน้อย

In [22]:
accidents_step4_loaded = pl.read_parquet(settings.staging_dir / "accidents.parquet")

**พิกัด (CLAUDE.md P7):** ตั้งค่า null ให้พิกัดที่อยู่นอกกรอบพิกัดประเทศไทย (lat 5-21, lon 97-106)
แล้วเพิ่ม `is_geo_valid` -- ไม่ลบแถวทิ้ง

In [23]:
accidents_geo, coord_log = clean_coordinates(accidents_step4_loaded)

cleaning_log.append(
    CleaningLogEntry(
        "5_coordinates",
        "clean_coordinates",
        coord_log.rows_total,
        coord_log.rows_is_geo_valid_false,
        f"lat out-of-bounds nulled={coord_log.rows_lat_out_of_bounds}, "
        f"lon out-of-bounds nulled={coord_log.rows_lon_out_of_bounds}, "
        f"is_geo_valid=false for {coord_log.rows_is_geo_valid_false} rows "
        "(post-dedup counts -- CLAUDE.md's 5/10 figures are pre-dedup, "
        "on the full 161,307-row file; a handful of the affected rows were "
        "themselves collapsed away by Step 4's dedup, which is why these "
        "counts are smaller)",
    )
)

pl.DataFrame(
    [
        {
            "rows_total": coord_log.rows_total,
            "rows_lat_out_of_bounds": coord_log.rows_lat_out_of_bounds,
            "rows_lon_out_of_bounds": coord_log.rows_lon_out_of_bounds,
            "rows_is_geo_valid_false": coord_log.rows_is_geo_valid_false,
        }
    ]
)

2026-08-14T03:06:54.996993Z [info     ] coordinates_cleaned            rows_is_geo_valid_false=801 rows_lat_out_of_bounds=5 rows_lon_out_of_bounds=8 rows_total=159605 run_id=2a5e957b166e45ff9677162dbb0db424


rows_total,rows_lat_out_of_bounds,rows_lon_out_of_bounds,rows_is_geo_valid_false
i64,i64,i64,i64
159605,5,8,801


**เวลา (CLAUDE.md P2):** แปลงทั้ง clock string (`"21:10"`) และ Excel day-fraction
(`"0.375"` = 09:00) เป็น `accident_time` เดียว ตรวจสอบค่า `"0:00"` (219 ครั้งใน raw file
ทั้งหมด) เทียบกับ `เวลาที่รายงาน` ก่อนตัดสินใจ

In [24]:
midnight_rows = accidents_geo.filter(pl.col("เวลา") == "0:00")
pl.DataFrame(
    {
        "rows_with_literal_midnight_time": [midnight_rows.height],
        "of_which_null_report_time": [midnight_rows.filter(pl.col("เวลาที่รายงาน").is_null()).height],
        "of_which_report_time_also_0_00": [
            midnight_rows.filter(pl.col("เวลาที่รายงาน") == "0:00").height
        ],
    }
)

rows_with_literal_midnight_time,of_which_null_report_time,of_which_report_time_also_0_00
i64,i64,i64
215,0,0


**ผลตรวจสอบ:** ทั้ง 215 แถว (หลัง dedup; ต้นฉบับ CLAUDE.md ระบุ 219 ก่อน dedup) มี
`เวลาที่รายงาน` ไม่ null และกระจายตัวปกติ ไม่มีตัวไหนเป็น `"0:00"` เช่นกัน -- ไม่มีหลักฐานว่าเป็น
placeholder ร่วมกับ `เวลาที่รายงาน` จึงไม่ null ค่านี้ทิ้ง (จะเป็นการเดาไปอีกทาง) แต่เพิ่ม flag
`time_is_literal_midnight` ไว้ให้ downstream กรองได้หากต้องการ

In [25]:
accidents_time, time_log = convert_times(accidents_geo)

cleaning_log.append(
    CleaningLogEntry(
        "5_time_format",
        "convert_times",
        time_log.rows_total,
        time_log.rows_unparseable,
        f"parsed={time_log.rows_parsed}, unparseable={time_log.rows_unparseable}, "
        f"literal_midnight_flagged={time_log.rows_literal_midnight}",
    )
)

pl.DataFrame(
    [
        {
            "rows_total": time_log.rows_total,
            "rows_parsed": time_log.rows_parsed,
            "rows_unparseable": time_log.rows_unparseable,
            "rows_literal_midnight": time_log.rows_literal_midnight,
        }
    ]
)

2026-08-14T03:06:55.081245Z [info     ] times_converted                rows_literal_midnight=215 rows_parsed=159605 rows_total=159605 rows_unparseable=0 run_id=2a5e957b166e45ff9677162dbb0db424


rows_total,rows_parsed,rows_unparseable,rows_literal_midnight
i64,i64,i64,i64
159605,159605,0,215


**Injury breakdown flag (CLAUDE.md P7):** `accident2019.csv` ไม่เคยเก็บ
`ผู้บาดเจ็บสาหัส`/`ผู้บาดเจ็บเล็กน้อย` แยก (null 100%) -- เพิ่ม `has_injury_breakdown = false`
สำหรับปีนั้นเท่านั้น ไม่ impute ค่าแยก และไม่แตะ `รวมจำนวนผู้บาดเจ็บ`/`ผู้เสียชีวิต`

In [26]:
accidents_step5, injury_log = add_injury_breakdown_flag(accidents_time)

cleaning_log.append(
    CleaningLogEntry(
        "5_injury_breakdown_flag",
        "add_injury_breakdown_flag",
        injury_log.rows_total,
        injury_log.rows_without_breakdown,
        f"has_injury_breakdown=false for {injury_log.rows_without_breakdown} rows "
        "(accident2019.csv)",
    )
)

pl.DataFrame(
    [
        {
            "rows_total": injury_log.rows_total,
            "rows_without_breakdown": injury_log.rows_without_breakdown,
        }
    ]
)

2026-08-14T03:06:55.093931Z [info     ] injury_breakdown_flagged       rows_total=159605 rows_without_breakdown=19194 run_id=2a5e957b166e45ff9677162dbb0db424


rows_total,rows_without_breakdown
i64,i64
159605,19194


In [27]:
accidents_step5.write_parquet(settings.staging_dir / "accidents.parquet")
"step5 written"

'step5 written'

## 6. Validate + write staging

โหลด `accidents_step5` (159,605 แถว, ผ่านทุกขั้นตอนแล้ว) มา validate: Gate 1/Gate 2,
ยืนยัน row-count conservation ตลอด pipeline, re-run pandera contract บน raw file เพื่อยืนยันว่า
baseline ของ Phase 1 ยังไม่เปลี่ยน, บันทึก cleaning_log.csv, สร้าง before/after day-of-month
histogram, แล้วเขียนตารางสุดท้ายลง staging

**Gate 1 (post-Extract, pre-dedup) บน `accidents_step1`, Gate 2 (post-Load, post-dedup)
บน `accidents_step5`** -- ทั้งคู่ต้องตรงตัวเลขใน CLAUDE.md เป๊ะๆ

In [28]:
gate1_results = check_gate1_reconciliation(accidents_step1)
gate2_results = check_gate2_reconciliation(accidents_step5)

for r in [*gate1_results, *gate2_results]:
    assert r.passed, f"{r.label} FAILED: expected {r.expected}, got {r.actual}"

pl.DataFrame(
    [
        {"label": r.label, "expected": r.expected, "actual": r.actual, "passed": r.passed}
        for r in [*gate1_results, *gate2_results]
    ]
)

label,expected,actual,passed
str,i64,i64,bool
"""gate1_accident_count""",161307,161307,true
"""gate1_fatality_count""",21814,21814,true
"""gate2_accident_count""",159605,159605,true
"""gate2_fatality_count""",21497,21497,true


**Re-run เต็มรูปแบบของ Phase 1 pandera contract บน `accidents_raw`** (ก่อนแก้อะไรเลย) --
ยืนยันว่าไฟล์ raw ยังตรงกับที่ Phase 1 โปรไฟล์ไว้ทุกตัวเลข ไม่มีอะไรเปลี่ยนไปโดยไม่รู้ตัว
(contract ตัวนี้ออกแบบมาให้รันกับ raw column ที่ยังไม่ผ่าน coalesce เท่านั้น เพราะ
`check_twin_column_pairs` ต้องการทั้งคอลัมน์เก่าและใหม่พร้อมกัน)

In [29]:
raw_report = run_accidents_raw_contract(accidents_raw)

assert raw_report.gate1[0].passed and raw_report.gate1[1].passed
assert all(t.passed for t in raw_report.twin_columns)
assert all(d.passed for d in raw_report.date_formats)
assert raw_report.province.passed

pl.DataFrame(
    [
        {
            "column_schema_failures": raw_report.column_schema_failure_cases.height,
            "province_orphans": len(raw_report.province.orphan_values),
            "injury_total_mismatches_raw": raw_report.injury_totals.mismatched_row_count,
        }
    ]
)

column_schema_failures,province_orphans,injury_total_mismatches_raw
i64,i64,i64
93,0,9872


**Injury-total mismatch บนข้อมูลที่ clean แล้ว (post-dedup)** -- ควรยังคงพบ mismatch
(ไม่ imputed ตามที่ตัดสินใจไว้) แต่จำนวนลดลงเล็กน้อยจาก dedup

In [30]:
injury_cleaned = check_injury_totals(accidents_step5)
pl.DataFrame(
    [
        {
            "mismatched_row_count": injury_cleaned.mismatched_row_count,
            "total_rows": injury_cleaned.total_rows,
            "mismatch_rate": round(injury_cleaned.mismatch_rate, 4),
        }
    ]
)

mismatched_row_count,total_rows,mismatch_rate
i64,i64,f64
9872,159605,0.0619


**Row-count conservation end-to-end:** ทุกแถวใน raw file (161,307) ต้องอยู่ในสถานะใด
สถานะหนึ่งเท่านั้น -- แถวที่เหลืออยู่ (159,605) หรือแถวที่ถูกยุบเข้ากลุ่ม duplicate ที่รู้จักแล้ว
(1,702 แถวส่วนเกิน จาก 1,641 กลุ่ม) ไม่มีแถวไหนหายไปเฉยๆ

In [31]:
rows_removed_by_dedup = dedup_result.rows_before - dedup_result.rows_after

pl.DataFrame(
    [
        {
            "raw_rows": accidents_step1.height,
            "final_rows": accidents_step5.height,
            "rows_removed_by_dedup": rows_removed_by_dedup,
            "reconciled": accidents_step1.height == accidents_step5.height + rows_removed_by_dedup,
        }
    ]
)

raw_rows,final_rows,rows_removed_by_dedup,reconciled
i64,i64,i64,bool
161307,159605,1702,true


**83-row date-parseability gap (P1/P7), ยืนยันซ้ำบนข้อมูลหลัง dedup:** 78 null
`วันที่รายงาน` (74 จาก accident2026.csv ที่ยังไม่ปิดปี + 4 จาก accident2023.csv) + 4 `"//"` ใน
accident2022.csv + 1 accident2020.csv Excel-serial stray (ที่ safety net #1 ยอมรับไว้แล้วตอน
checkpoint Step 3) = 83 พอดี

In [32]:
null_report_date = accidents_step5.filter(pl.col("report_date").is_null()).height
null_accident_date = accidents_step5.filter(pl.col("accident_date").is_null()).height

pl.DataFrame(
    [
        {
            "null_report_date": null_report_date,
            "null_accident_date": null_accident_date,
            "total_gap": null_report_date + null_accident_date,
            "matches_83": null_report_date + null_accident_date == 83,
        }
    ]
)

null_report_date,null_accident_date,total_gap,matches_83
i64,i64,i64,bool
82,1,83,true


**Before/after day-of-month histogram** -- "before" คือ parse วันที่ด้วยสมมติฐาน
รูปแบบเดียว (`mdy`) ทั้งหมดทุกไฟล์ (สิ่งที่จะเกิดถ้าไม่ตรวจ per-file format เลย), "after" คือ
`accident_date` จริงจาก Step 3 ที่ตรวจ format ต่อไฟล์ -- บันทึกเป็น PNG ลง `reports/`
สำหรับใส่ในรายงาน

In [33]:
naive_parsed = parse_dates(
    accidents_step1,
    file_date_formats=dict.fromkeys(SOURCE_FILE_DATE_FORMAT, "mdy"),
)
before_hist = day_of_month_histogram(naive_parsed, accident_date_column="accident_date")
after_hist = day_of_month_histogram(accidents_step5, accident_date_column="accident_date")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].bar(list(before_hist.keys()), list(before_hist.values()), color="firebrick")
axes[0].set_title("Before: naive single-format (mdy) parse")
axes[0].set_xlabel("day of month")
axes[1].bar(list(after_hist.keys()), list(after_hist.values()), color="seagreen")
axes[1].set_title("After: per-file format-aware parse (Step 3)")
axes[1].set_xlabel("day of month")
fig.tight_layout()
fig.savefig(settings.reports_dir / "day_of_month_histogram_before_after.png", dpi=150)
plt.close(fig)
"saved to reports/day_of_month_histogram_before_after.png"

'saved to reports/day_of_month_histogram_before_after.png'

**เขียนตารางสุดท้าย** -- `accidents_step5` คือสิ่งที่อยู่ใน `accidents.parquet` แล้ว
(เขียนไว้ท้าย Step 5) ยืนยันด้วยการเปรียบเทียบ hash ว่าตรงกับไฟล์บนดิสก์

In [34]:
on_disk = pl.read_parquet(settings.staging_dir / "accidents.parquet")
assert on_disk.equals(accidents_step5), "staged accidents.parquet does not match accidents_step5"
"confirmed: accidents.parquet on disk matches the final in-memory Step 5 result"

'confirmed: accidents.parquet on disk matches the final in-memory Step 5 result'

**เขียน `reports/cleaning_log.csv`** จาก `cleaning_log` ที่สะสมไว้ตลอด notebook (N4)

In [35]:
write_cleaning_log(cleaning_log, str(settings.reports_dir / "cleaning_log.csv"))
cleaning_log_to_frame(cleaning_log)

step,rule,rows_before,rows_affected,description
str,str,i64,i64,str
"""1_extract""","""coalesce_twin_columns:รถยนต์นั…",161307,8738,"""รถยนต์นั่งส่วนบุคคล <- รถยนต์น…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกไม่เกิน10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถบรรทุก…",161307,8738,"""รถบรรทุกมากกว่า10ล้อ <- รถบรรท…"
"""1_extract""","""coalesce_twin_columns:รถอื่นๆ""",161307,8738,"""รถอื่นๆ <- อื่นๆ: 133375 from …"
"""1_extract""","""pivot_rainfall_leads""",30954,5159,"""30954 rows (6 leads x group) -…"
…,…,…,…,…
"""2_taxonomy""","""apply_taxonomy:บริเวณที่เกิดเห…",161307,8462,"""mapped=8462 null=152845 unmapp…"
"""4_dedup""","""deduplicate_accidents""",161307,1702,"""1641 ACC_CODE groups collapsed…"
"""5_coordinates""","""clean_coordinates""",159605,801,"""lat out-of-bounds nulled=5, lo…"


**ยืนยันครั้งสุดท้าย: ไม่มีการแก้ไขไฟล์ใดใน `data/00_raw/`** (N1)

In [36]:
import subprocess

result = subprocess.run(
    ["git", "status", "--short", "data/00_raw/"],
    cwd=settings.base_dir,
    capture_output=True,
    text=True,
    check=True,
)
assert result.stdout == "", f"data/00_raw/ was modified:\n{result.stdout}"
"confirmed: data/00_raw/ untouched"

'confirmed: data/00_raw/ untouched'